# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from tools import get_gemini_client

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
gemini = get_gemini_client()

API key looks good so far


In [3]:
links = fetch_website_links("https://www.heise.de/")
links

['/',
 'https://www.heise.de/plus/abo?affiliateId=30165&wt_mc=intern.abo.plus.hp_ea_2025.ho_navilink.desktop.desktop',
 '/plus',
 '/ct',
 '/ix',
 '/foto',
 '/mac-and-i',
 '/make',
 '/select/',
 '/newsticker/',
 '/hintergrund',
 '/ratgeber',
 '/tests',
 '/meinung',
 '/plus',
 'https://www.telepolis.de',
 '/autos',
 '/bestenlisten',
 '/tipps-tricks',
 'https://shop.heise.de/',
 'https://jobs.heise.de/',
 'https://heise-academy.de/',
 '/download/',
 'https://preisvergleich.heise.de/',
 '/tarifrechner',
 'https://compaliate.heise.de/',
 'https://shop.heise.de/zeitschriften-abo/',
 'https://www.heise.de/meinabo',
 '/tools',
 '/netze/netzwerk-tools/imonitor-internet-stoerungen/',
 '/loseblattwerke/',
 'https://spiele.heise.de/',
 'https://www.heise-gruppe.de/artikel/heise-medien-3904998.html',
 'https://www.heise-regioconcept.de/',
 'https://business-services.heise.de/',
 'https://www.heisegroup.de',
 'https://mediadaten.heise.de/',
 'https://www.heise-gruppe.de/artikel/heise-als-Arbeitgeber

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages. Distinguish between irrelevant and relevant links for the brochure.
Include relative links to your response. Expand relative links to absolute links.
You must respond in strict JSON without any additional text as in this example:

{
    "relevant": {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "careers page", "url": "https://another.full.url/careers"}
        ]
    },
    "irrelevant": {
        "links": [
            {"type": "privacy policy", "url": "https://full.url/goes/here/privacy-policy"},
            {"type": "foreign link not closly related", "url": "https://another.url/with/no/relevance"}
            
        ]
    }
}
"""

link_user_prompt = """
You are provided with a list of links found on a webpage. Filter out irrelevant links and return the relevant links in strict JSON format.
"""

In [4]:
def get_links_user_prompt(url):
    links = fetch_website_links(url)
    user_prompt = f"""
Here is the list of links on the website {url} 
Links:

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://www.heise.de/"))

In [5]:
def select_relevant_links(url):
    response = gemini.chat.completions.create( #pyrefly: ignore
        model="gemini-flash-latest",#gemini-3-pro-preview"
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    #links = json.loads(result)
    return result
    

In [6]:
result = select_relevant_links("https://roeckenhof.de/")

In [7]:
print(result)

{
    "relevant": {
        "links": [
            {
                "type": "company description/location",
                "url": "https://roeckenhof.de/roeckenhof/"
            },
            {
                "type": "about page",
                "url": "https://roeckenhof.de/ueber-uns/"
            },
            {
                "type": "membership/joining",
                "url": "https://roeckenhof.de/mitglied-werden/"
            },
            {
                "type": "news/blog",
                "url": "https://roeckenhof.de/blog/"
            },
            {
                "type": "cafe services",
                "url": "https://roeckenhof.de/cafe/"
            },
            {
                "type": "cafe services detail",
                "url": "https://roeckenhof.de/dorfcafe/"
            },
            {
                "type": "general image gallery",
                "url": "https://roeckenhof.de/bilder/"
            },
            {
                "type": "conta

In [8]:
results_json = json.loads(result)
links_only = [l['url'] for l in results_json['relevant']['links']]
print(links_only)

['https://roeckenhof.de/roeckenhof/', 'https://roeckenhof.de/ueber-uns/', 'https://roeckenhof.de/mitglied-werden/', 'https://roeckenhof.de/blog/', 'https://roeckenhof.de/cafe/', 'https://roeckenhof.de/dorfcafe/', 'https://roeckenhof.de/bilder/', 'https://roeckenhof.de/kontakt/']


In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result  = response.choices[0].message.content or ''
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = results_json['relevant']['links']
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link['url'])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://roeckenhof.de/"))

## Landing Page:

https://roeckenhof.de/

Zum Inhalt springen
Start
Röckenhof
Unser Verein
Mitglied werden
Veranstaltungen
Café
Berichte Archiv
Bilder
Suchen nach:
Suchen
Suchen
Main Menu
Start
Röckenhof
Unser Verein
Mitglied werden
Veranstaltungen
Café
Berichte Archiv
Bilder
Über das Menü oben rechts gelangt Ihr auf andere nützliche Informationen.
Dorferneuerung Röckenhof
Mitdenken – Mitreden – Mitgestalten
Unser Verein
Wir sind die Dorferneuerung Röckenhof
Röckenhof ist ein Ortsteil der Gemeinde Kalchreuth
im Landkreis Erlangen/Höchstadt,
inmitten von Kirschgärten
am Rande des Nürnberger Reichswaldes.
Ziel unseres Vereines ist es, die Projekte und Vorhaben der Dorferneuerung nachhaltig und langfristig zu sichern.
Unser Verein ist als gemeinnützig anerkannt.
Veranstaltungen / Termine
Mitdenken – Mitreden – Mitgestalten
Sonnenwendfeuer 2025
Sonnenwendfeuer 2025 Sonnenwendfeuer 2025 in Röckenhof – Ein Fest der GemeinschaftAm Freitag, den 20. Juni…
Weiterlesen
Boule-Turnier
Boule-Turnier

In [16]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:100_000] # Truncate if more than 5,000 characters
    return user_prompt

In [98]:
get_brochure_user_prompt("heise.de", "https://www.heise.de/")

'\nYou are looking at a company called: heise.de\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nheise online - IT-News, Nachrichten und Hintergründe | heise online\n\nheise+ entdecken\nSuchen\nAbo\nSuchen\nAlle Magazine im Browser lesen\nIT News\nNewsticker\nHintergründe\nRatgeber\nTestberichte\nMeinungen\nOnline-Magazine\nheise\n+\nTelepolis\nheise autos\nbestenlisten\ntipps+tricks\nServices\nheise shop\nheise jobs\nheise academy\nheise download\nheise preisvergleich\nTarifrechner\nheise compaliate\nAbo bestellen\nMein Abo\nNetzwerktools\niMonitor\nLoseblattwerke\nSpiele\nÜber uns\nheise medien\nheise regioconcept\nheise business services\nSponsoring\nMediadaten\nKarriere\nPresse\nAnzeige\nThemenspecial ChromeOS\nNewsletter\nheise-Bot\nPush\n-Nachrichten\nNewsticker\nSecurity\nIT & Tech\nDeveloper\nKI\nEntertainment\nWissenschaft\nBestenlisten\

In [99]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3-pro-preview",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3-pro-preview",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )
    response = ''
    display_handle = display(Markdown(''), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [18]:
stream_brochure("roeckehof.de", "https://roeckenhof.de/")

# Röckenhof: Think Along – Speak Up – Shape Together

Welcome to **Dorferneuerung Röckenhof e.V.**, the driving force behind the vibrant village life of Röckenhof. Located amidst picturesque cherry orchards on the edge of the Nuremberg Imperial Forest, we are a community-focused non-profit dedicated to preserving our heritage while building a dynamic future.

***

### 🏡 Who We Are & Our Culture
Röckenhof is more than just a district of Kalchreuth; it is a community with roots dating back to the 12th century. Our association was founded to ensure that our village does not become a mere "dormitory town."

**Our Mission:**
*   **Preservation:** safeguarding rural settlement forms, cultural customs, and the village lifestyle.
*   **Integration:** Fostering harmony between long-established families and new residents.
*   **Action:** We believe in "Mitdenken – Mitreden – Mitgestalten" (Thinking along, speaking up, shaping together).

**Community Spirit:**
Our pride and joy is the **Hirtenhaus (Shepherd’s House)**. Once a derelict sandstone building, it was saved from decay and renovated into a sparkling community center through over **3,000 voluntary working hours** by our members. This spirit of hands-on cooperation defines our culture.

***

### ☕ For Our Community & Visitors
We offer a welcoming atmosphere for residents, hikers, and visitors from the Nuremberg-Fürth-Erlangen metropolitan area.

**The Hirtenhaus Café**
Our "Dorfcafé" is the heart of social life in Röckenhof.
*   **Open:** Thursdays and Fridays, 14:00 – 18:00 (closed August/December).
*   **Offerings:** Hand-filtered coffee, tea, and homemade cakes at family-friendly prices.
*   **Ambience:** Cozy interior with dimmed lighting and outdoor seating for sunny days. Includes free WiFi and a selection of books and local newspapers.

**Signature Events**
We organize events that bring generations together:
*   **Summer Solstice Bonfire:** A magical evening with a torchlight procession for children.
*   **Autumn Festival:** Featuring our famous **Bobby Car Race** and traditional apple juice pressing.
*   **New Year's Eve:** Fireworks and celebrations at the village pond (Dorfweiher).
*   **Community Activities:** Boule tournaments, table foosball nights, senior meet-ups (2nd Wednesday of the month), and children's cinema.

***

### 🤝 Join the Team: Membership & Volunteering
While we do not offer traditional corporate jobs, we offer fulfilling opportunities to engage with your neighbors and make a tangible difference in your local environment. We are looking for individuals who want to be part of an active "Team Röckenhof."

**Why Join?**
*   **Impact:** Help shape the physical and social landscape of the village (e.g., maintaining the pond, decorating the Easter fountain).
*   **Connection:** Meet new friends and integrate quickly into village life.
*   **Flexibility:** Whether you want to bake a cake, help set up for a festival, or take on a board position—every contribution is valued.

**We are looking for:**
*   Event organizers and helpers.
*   Bakers and service staff for the Dorfcafé.
*   People with ideas for lectures, courses, or new projects.

***

### 📍 Contact & Location

We invite you to visit us, enjoy a piece of cake, or attend our next town meeting.

**Location:**
Birkgartenstraße 1
90562 Kalchreuth (Ortsteil Röckenhof)

**Connect with us:**
*   **Phone:** 0911 9569417
*   **Email:** info(at)roeckenhof.de
*   **Website:** roeckenhof.de

**Come for the nature, stay for the community.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>